In [1]:
# =====================================================================
import os, sys, re, gc, math, glob, time, random, warnings, subprocess, importlib, contextlib

os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")


def _ensure(pip_name, import_name):
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=False)


for _p, _i in [("transformers", "transformers"), ("scikit-learn", "sklearn"),
               ("pandas", "pandas"), ("scipy", "scipy")]:
    _ensure(_p, _i)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from scipy.optimize import minimize
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (f1_score, accuracy_score, precision_score,
                             recall_score, log_loss)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"

# =====================================================================
# CONFIGURATION  (edit here)
# =====================================================================
CFG = dict(
    dataset="multi-lang",            # "ghadhab" or "multi-lang"
    data_path=None,               # explicit CSV path (optional)
    strict_dropna=False,          # True = also drop rows with missing message/diff (old behaviour)
    out_dir=None,                 # where to save the CSV (default: /kaggle/working or cwd)

    # ---- encoders (any HF checkpoint that AutoModel can load) ----
    msg_encoders=["sentence-transformers/all-mpnet-base-v2",
                  "sentence-transformers/paraphrase-mpnet-base-v2"],
    code_encoder="microsoft/unixcoder-base",
    ft_backbones=["microsoft/unixcoder-base",
                  "sentence-transformers/all-mpnet-base-v2"],   # fine-tuned in FULL mode
    con_backbone="sentence-transformers/paraphrase-mpnet-base-v2",  # contrastive FT in FEW-SHOT mode
    msg_max_len=96, diff_max_len=320, joint_max_len=384, enc_bs=32, diff_chars=1200,

    # ---- full supervision ----
    full_ft=True, ft_seeds=[42, 43, 44], ft_epochs=8, ft_lr=3e-5, ft_bs=16, ft_max_len=384,
    lr_grid=[1, 4, 16, 64],

    # ---- few-shot ----
    shots=[1, 5, 10, 15, 20], fs_seeds=[42, 43, 44, 45, 46],
    fs_contrastive=True, fs_con_lr=3e-5, fs_con_bs=16, fs_con_max_len=320, fs_con_weight=2.0,
    tau=25.0,            # softmax temperature of the prototype head
    fs_C=10.0,           # LR strength for tiny support sets (old code: 1.0 = badly under-fit)
    anchor_mass=1.5,     # total weight of definitional anchors per class at K=1 (decays ~1/sqrt(K))
    lex_weight=0.6, lex_beta=1.2,   # keyword-lexicon prior
    self_train=False,    # OPTIONAL semi-supervised step on the *unlabelled* train pool
    self_train_per_class=10, self_train_w=0.5,

    # ---- optional: put the numbers you must beat, e.g. {"Full": 80.06, "5-Shot": 70.0}
    sota_f1={},
)

CLASSES = ["Corrective", "Adaptive", "Perfective"]
LABEL_TO_ID = {c: i for i, c in enumerate(CLASSES)}
NC = 3

LABEL_MAP = {
    'c': 'Corrective', 'corrective': 'Corrective', '1': 'Corrective',
    'a': 'Adaptive', 'adaptive': 'Adaptive', '0': 'Adaptive',
    'p': 'Perfective', 'perfective': 'Perfective', '2': 'Perfective',
}

# Swanson-style definitional anchors (unified format, as in the original code)
CLASS_DEFINITIONAL_ANCHORS = {
    'Corrective': "[STRUCTURAL INTENT]: [COND_OR_ERROR_HANDLING], [BUG_CORRECTION_SEMANTICS]\n[COMMIT MESSAGE]: Fix runtime bug, resolve null pointer exception, repair defect, and handle errors.\n[DISTILLED CODE DIFF]:\n- if (target == null)\n+ if (target != null && target.isValid())\n+ catch (Exception e) { log.error(e); }",
    'Adaptive': "[STRUCTURAL INTENT]: [NEW_DECLARATION_OR_API], [NEW_FILE_CREATED]\n[COMMIT MESSAGE]: Add new feature, implement public API interface, export new functionality, and extend module.\n[DISTILLED CODE DIFF]:\n+ public interface NewServiceAPI {\n+     void registerExtension();\n+ }",
    'Perfective': "[STRUCTURAL INTENT]: [DOC_OR_CONFIG_MODIFIED], [LOCAL_REFACTORING_OR_RENAME]\n[COMMIT MESSAGE]: Refactor internal code structure, clean up dead code, optimize design, and update documentation.\n[DISTILLED CODE DIFF]:\n- int old_counter = 0;\n+ int cleanIndex = 0;\n// Optimize internal memory layout",
}

# Generic pseudo commit messages (definitions only - no dataset text)
PSEUDO_MESSAGES = {
    'Corrective': ["fix bug in null pointer handling", "fixed crash when the file is missing",
                   "resolve exception thrown on invalid input", "correct wrong result in calculation",
                   "fix failing test case", "patch memory leak", "handle error case that caused failure",
                   "repair broken behaviour after regression"],
    'Adaptive': ["add new feature to support additional format", "implement new API endpoint",
                 "introduce support for a new platform", "create new class for the extension mechanism",
                 "add option to configure new behaviour", "upgrade dependency to a newer version",
                 "extend module with new functionality", "port code to the new library version"],
    'Perfective': ["refactor code to improve readability", "clean up unused code and imports",
                   "rename variables for clarity", "update documentation and comments",
                   "optimize performance of the internal loop", "simplify method structure",
                   "reformat code style", "reorganize package structure"],
}

LEXICON = {
    'Corrective': [r"\bfix(es|ed|ing)?\b", r"\bbugs?(fix)?\b", r"\b(error|errors|fail|fails|failed|failure|crash\w*|exception|npe)\b",
                   r"\b(issue|issues|defect|problem|incorrect\w*|wrong|broken|regression|leak\w*|resolv\w+|repair\w*|patch\w*|hotfix|workaround|mistake|revert\w*)\b",
                   r"\bnull ?pointer\b"],
    'Adaptive': [r"\badd(s|ed|ing)?\b", r"\bnew\b", r"\bimplement\w*", r"\bsupport(s|ed|ing)?\b", r"\bintroduc\w+",
                 r"\bcreate[sd]?\b", r"\benable[sd]?\b", r"\bextend\w*", r"\ballow(s|ed)?\b", r"\bupgrad\w+",
                 r"\bmigrat\w+", r"\bport(ed|ing)?\b", r"\bintegrat\w+", r"\bfeature\w*"],
    'Perfective': [r"\brefactor\w*", r"\bclean(s|ed|ing)?( ?up)?\b", r"\brenam\w+", r"\bmov(e|ed|ing)\b", r"\bsimplif\w+",
                   r"\boptimi[sz]\w+", r"\bimprov\w+", r"\bperformance\b", r"\bdocs?\b", r"\bdocumentation\b", r"\bjavadoc\b",
                   r"\bcomments?\b", r"\bformat\w*", r"\bstyle\b", r"\bunused\b", r"\bdeprecat\w+", r"\breorgani[sz]\w+",
                   r"\brestructur\w+", r"\breadme\b", r"\bwhitespace\b", r"\blicen[sc]e\b", r"\btypos?\b", r"\bremov(e|ed|ing)\b",
                   r"\bcosmetic\b", r"\btidy\b", r"\bredundan\w+"],
}
LEX_RE = {c: [re.compile(p, re.I) for p in ps] for c, ps in LEXICON.items()}


# =====================================================================
# GENERIC HELPERS
# =====================================================================
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def free():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _s(x):
    if x is None or (not isinstance(x, str) and pd.isna(x)):
        return ""
    return str(x)


def l2n(X):
    return X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-8)


def softmax_np(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def metrics(y, p):
    return {"Accuracy": accuracy_score(y, p),
            "Precision": precision_score(y, p, average="macro", zero_division=0),
            "Recall": recall_score(y, p, average="macro", zero_division=0),
            "F1-score": f1_score(y, p, average="macro", zero_division=0)}


def amp():
    return torch.autocast(device_type="cuda", dtype=torch.float16) if USE_AMP else contextlib.nullcontext()


def make_scaler():
    if not USE_AMP:
        return None
    try:
        return torch.amp.GradScaler("cuda")
    except Exception:
        return torch.cuda.amp.GradScaler()


def backward_step(loss, opt, sch, scaler, params, clip=1.0):
    if scaler is not None:
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, clip)
        scaler.step(opt)
        scaler.update()
    else:
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, clip)
        opt.step()
    sch.step()
    opt.zero_grad(set_to_none=True)


def mean_pool(h, mask):
    m = mask.unsqueeze(-1).to(h.dtype)
    return (h * m).sum(1) / m.sum(1).clamp(min=1e-6)


def embed_batch(model, tok, texts, max_len):
    enc = tok(texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(DEVICE)
    with amp():
        h = model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"]).last_hidden_state
    return mean_pool(h.float(), enc["attention_mask"])


@torch.inference_mode()
def encode_texts(model, tok, texts, max_len, bs):
    model.eval()
    texts = list(texts)
    if not texts:
        return np.zeros((0, model.config.hidden_size), dtype=np.float32)
    order = np.argsort([len(t) for t in texts])
    chunks = []
    for i in range(0, len(texts), bs):
        idx = order[i:i + bs]
        chunks.append(embed_batch(model, tok, [texts[j] for j in idx], max_len).cpu().numpy())
    E = np.concatenate(chunks)
    out = np.zeros_like(E)
    out[order] = E
    return out


# =====================================================================
# 1. TEXT / STRUCTURE EXTRACTION
# =====================================================================
def clean_msg(m):
    m = _s(m).replace("\r", " ")
    return re.sub(r"\s+", " ", m).strip()[:400]


def _diff_parts(d):
    add = re.findall(r"^\+(?!\+)(.*)$", d, re.M)
    rem = re.findall(r"^-(?!-)(.*)$", d, re.M)
    files = re.findall(r"^\+\+\+\s+(?:b/)?(.*)$", d, re.M)
    files += re.findall(r"^diff --git a/(\S+)", d, re.M)
    files = [f.strip() for f in dict.fromkeys(files) if f.strip() and f.strip() != "/dev/null"]
    return add, rem, files


def _tags(msg, d, add, rem, files):
    added_text, removed_text = " ".join(add), " ".join(rem)
    tags = []
    if re.search(r"\b(if|catch|try|except|throw|throws|assert|null|nil|None)\b", added_text):
        tags.append("[COND_OR_ERROR_HANDLING]")
    if re.search(r"\b(fix|bug|issue|crash|error|fail|leak)\b", msg + " " + added_text + " " + removed_text, re.I):
        tags.append("[BUG_CORRECTION_SEMANTICS]")
    if re.search(r"\b(class|def|function|interface|public|export|import|include|new)\b", added_text):
        tags.append("[NEW_DECLARATION_OR_API]")
    if "new file mode" in d:
        tags.append("[NEW_FILE_CREATED]")
    if any(f.lower().endswith(('.md', '.txt', '.json', '.yml', '.yaml', '.xml')) for f in files):
        tags.append("[DOC_OR_CONFIG_MODIFIED]")
    if add and rem and abs(len(add) - len(rem)) <= 3:
        tags.append("[LOCAL_REFACTORING_OR_RENAME]")
    return tags


def build_parts(msg, diff):
    m = clean_msg(msg)
    d = _s(diff)
    add, rem, files = _diff_parts(d)
    tags = _tags(m, d, add, rem, files)
    exts = list(dict.fromkeys([os.path.splitext(f)[1] for f in files if os.path.splitext(f)[1]]))
    ext_str = ", ".join(exts)[:40] if exts else "none"
    intent = f"[STRUCTURAL INTENT]: {', '.join(tags) if tags else '[GENERAL_MODIFICATION]'} (exts: {ext_str})"

    lines = [l[:120] for l in d.splitlines()
             if l.startswith(("+", "-")) and not l.startswith(("+++", "---"))][:30]
    if not lines:   # diff not in +/- format: fall back to the first raw lines
        lines = [l.strip()[:120] for l in d.splitlines() if l.strip()][:30]
    dtxt = "\n".join(lines)
    if files:
        dtxt = "FILES: " + ", ".join(files[:5]) + "\n" + dtxt
    dtxt = dtxt[:CFG["diff_chars"]]

    unified = (f"{intent}\n[COMMIT MESSAGE]: {m or '[NO_MESSAGE]'}\n"
               f"[DISTILLED CODE DIFF]:\n{dtxt or '[NO_DIFF]'}")
    unified_nodiff = (f"[STRUCTURAL INTENT]: [GENERAL_MODIFICATION]\n[COMMIT MESSAGE]: {m or '[NO_MESSAGE]'}\n"
                      f"[DISTILLED CODE DIFF]:\n[NO_DIFF]")
    return dict(m=m, intent=intent, dtxt=dtxt, unified=unified, unified_nodiff=unified_nodiff)


def lex_counts(msg):
    m = clean_msg(msg)
    return np.array([min(3, sum(len(r.findall(m)) for r in LEX_RE[c])) for c in CLASSES], dtype=float)


def lex_probs_from_counts(cnt, beta):
    p = np.exp(beta * cnt)
    p[cnt.sum(axis=1) == 0] = 1.0
    return p / p.sum(axis=1, keepdims=True)


STRUCT_EXTS = ['.java', '.py', '.js', '.c', '.cpp', '.h', '.md', '.txt', '.xml', '.json',
               '.yml', '.yaml', '.gradle', '.properties', '.html', '.css']


def struct_features(msg, diff):
    m = clean_msg(msg); ml = m.lower(); d = _s(diff)
    add, rem, files = _diff_parts(d)
    exts = [os.path.splitext(f)[1].lower() for f in files]
    na, nr = len(add), len(rem)
    f = [math.log1p(na), math.log1p(nr), math.log1p(len(files)), (na - nr) / (na + nr + 1.0),
         float(na > 0 and nr == 0), float(nr > 0 and na == 0), float(na > 0 and nr > 0 and abs(na - nr) <= 3),
         float("new file mode" in d), float("deleted file mode" in d),
         float("rename from" in d or "similarity index" in d)]
    f += [float(e in exts) for e in STRUCT_EXTS]
    f.append(float(any("test" in x.lower() for x in files)))
    f.append(float(len(files) > 0 and all("test" in x.lower() for x in files)))
    nz = [l.strip() for l in add + rem if l.strip()]
    is_c = lambda l: l.startswith(("//", "/*", "*", "#", '"""', "'''", "<!--"))
    f.append(sum(is_c(l) for l in nz) / len(nz) if nz else 0.0)
    f.append(float(len(nz) > 0 and all(is_c(l) for l in nz)))
    norm = lambda L: re.sub(r"\s+", "", "".join(L))
    f.append(float(na > 0 and nr > 0 and norm(add) == norm(rem)))
    f.append(float(len(nz) > 0 and all(l.startswith(("import ", "from ", "#include", "package ")) for l in nz)))
    tags = _tags(m, d, add, rem, files)
    for t in ["[COND_OR_ERROR_HANDLING]", "[BUG_CORRECTION_SEMANTICS]", "[NEW_DECLARATION_OR_API]",
              "[NEW_FILE_CREATED]", "[DOC_OR_CONFIG_MODIFIED]", "[LOCAL_REFACTORING_OR_RENAME]"]:
        f.append(float(t in tags))
    f += [math.log1p(len(m.split())), float(bool(re.search(r"#\d+|\b[A-Z]{2,}-\d+\b", m))),
          float("merge" in ml)]
    f += list(np.log1p(lex_counts(m)))
    return f


def code_tokens(text):
    out = []
    for t in re.findall(r"[A-Za-z][A-Za-z0-9]*", text):
        out += re.sub(r"([a-z0-9])([A-Z])", r"\1 \2", t).lower().split()
    return out


def diff_token_doc(dtxt):
    a = [l[1:] for l in dtxt.splitlines() if l.startswith("+")]
    r = [l[1:] for l in dtxt.splitlines() if l.startswith("-")]
    return " ".join(["a_" + t for t in code_tokens(" ".join(a))] + ["d_" + t for t in code_tokens(" ".join(r))])


def anchors_for(kind):
    texts, labels = [], []
    if kind == "msg":
        for c in CLASSES:
            for t in PSEUDO_MESSAGES[c]:
                texts.append(t); labels.append(LABEL_TO_ID[c])
    elif kind == "diff":
        for c in CLASSES:
            texts.append(CLASS_DEFINITIONAL_ANCHORS[c].split("[DISTILLED CODE DIFF]:\n", 1)[1]); labels.append(LABEL_TO_ID[c])
    else:
        for c in CLASSES:
            texts.append(CLASS_DEFINITIONAL_ANCHORS[c]); labels.append(LABEL_TO_ID[c])
    return texts, np.array(labels)


# =====================================================================
# 2. DATA LOADING  (same 70/15/15, random_state=42 split as BooCC setup)
# =====================================================================
def normalize_label(v):
    s = _s(v).strip().lower()
    if s in LABEL_MAP:
        return LABEL_MAP[s]
    try:
        s2 = str(int(float(s)))
        if s2 in LABEL_MAP:
            return LABEL_MAP[s2]
    except Exception:
        pass
    for pref, name in (("corr", "Corrective"), ("adap", "Adaptive"), ("perf", "Perfective")):
        if s.startswith(pref):
            return name
    return None


def _pick_col(df, exact, contains, exclude=(), text_only=False):
    low = {c: str(c).lower().strip() for c in df.columns}
    ok = lambda c: c not in exclude and not (text_only and pd.api.types.is_numeric_dtype(df[c]))
    for name in exact:
        for c, l in low.items():
            if l == name and ok(c):
                return c
    for c, l in low.items():
        if ok(c) and any(sub in l for sub in contains):
            return c
    return None


def load_dataset_by_name_strict(dataset_name="ghadhab"):
    name = dataset_name.lower()
    if CFG["data_path"]:
        paths, globs = [CFG["data_path"]], []
    elif name == "ghadhab":
        paths = ["/kaggle/input/datasets/zohrehsadatsajjadi/ghadhab/Ghadhab.csv",
                 "/kaggle/input/ghadhab/Ghadhab.csv", "./Ghadhab.csv"]
        globs = ["/kaggle/input/**/Ghadhab*.csv", "/kaggle/input/**/ghadhab*.csv"]
    else:
        paths = ["/kaggle/input/datasets/zohrehsadatsajjadi/multi-lang-annotated/multi-lang-annotated.csv",
                 "/kaggle/input/multi-lang-annotated/multi-lang-annotated.csv", "./multi-lang-annotated.csv"]
        globs = ["/kaggle/input/**/multi-lang*.csv"]
    file_path = next((p for p in paths if os.path.exists(p)), None)
    if not file_path:
        for g in globs:
            hits = sorted(glob.glob(g, recursive=True))
            if hits:
                file_path = hits[0]; break
    if not file_path:
        raise FileNotFoundError(f"{dataset_name} dataset not found. Set CFG['data_path'] to the CSV.")

    try:
        raw = pd.read_csv(file_path, low_memory=False, on_bad_lines="skip")
    except UnicodeDecodeError:
        raw = pd.read_csv(file_path, low_memory=False, on_bad_lines="skip", encoding="latin-1")

    msg_col = _pick_col(raw, ["message", "msg", "commit_message", "commit message", "commit_msg", "text"], ["msg", "message"], text_only=True)
    lab_col = _pick_col(raw, ["label", "type", "class", "category"], ["label", "type", "class", "categ"], exclude=[msg_col])
    diff_col = _pick_col(raw, ["diff", "patch", "git_diff", "code_change", "changes"], ["diff", "patch", "change"],
                         exclude=[msg_col, lab_col], text_only=True)
    if msg_col is None: msg_col = raw.columns[0]
    if lab_col is None: lab_col = raw.columns[-1]
    if diff_col is None:   # longest remaining text column
        rest = [c for c in raw.columns if c not in (msg_col, lab_col) and not pd.api.types.is_numeric_dtype(raw[c])]
        if rest:
            diff_col = max(rest, key=lambda c: raw[c].astype(str).str.len().mean())
    print(f"[data] file={file_path}\n[data] columns -> message='{msg_col}', diff='{diff_col}', label='{lab_col}'")

    df = pd.DataFrame({"message": raw[msg_col],
                       "diff": raw[diff_col] if diff_col is not None else "",
                       "label_raw": raw[lab_col]})
    df["label"] = df["label_raw"].map(normalize_label)
    bad = df[df["label"].isna()]
    if len(bad):
        print(f"[data] WARNING: {len(bad)} rows have unrecognised labels, e.g. {bad['label_raw'].astype(str).unique()[:8]}")
    df = df.dropna(subset=["label"])
    if CFG["strict_dropna"]:
        df = df.dropna(subset=["message", "diff"])
    df["message"] = df["message"].map(_s)
    df["diff"] = df["diff"].map(_s)
    df = df.reset_index(drop=True)
    print(f"\n--- {dataset_name.upper()} loaded: {len(df)} commits | {df['label'].value_counts().to_dict()} ---")

    train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42)
    val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)
    parts = []
    for sp, part in (("train", train_df), ("val", val_df), ("test", test_df)):
        part = part.reset_index(drop=True).copy()
        part["split"] = sp
        parts.append(part)
    all_df = pd.concat(parts).reset_index(drop=True)
    all_df["y"] = all_df["label"].map(LABEL_TO_ID).astype(int)
    print(f"[data] split sizes: train={len(parts[0])} val={len(parts[1])} test={len(parts[2])}")
    return all_df


def add_text_fields(df):
    P = [build_parts(m, d) for m, d in zip(df["message"], df["diff"])]
    for key in ["m", "intent", "dtxt", "unified", "unified_nodiff"]:
        df[key] = [p[key] for p in P]
    return df


# =====================================================================
# 3. FROZEN MULTI-VIEW EMBEDDINGS
# =====================================================================
def view_specs():
    v = {}
    v["msg_all"] = dict(model=CFG["msg_encoders"][0], kind="msg", max_len=CFG["msg_max_len"])
    if len(CFG["msg_encoders"]) > 1:
        v["msg_para"] = dict(model=CFG["msg_encoders"][1], kind="msg", max_len=CFG["msg_max_len"])
    v["diff_code"] = dict(model=CFG["code_encoder"], kind="diff", max_len=CFG["diff_max_len"])
    v["joint_code"] = dict(model=CFG["code_encoder"], kind="unified", max_len=CFG["joint_max_len"])
    return v


def compute_frozen_views(df):
    texts = {"msg": df["m"].tolist(),
             "diff": [t if t.strip() else "[NO_DIFF]" for t in df["dtxt"]],
             "unified": df["unified"].tolist()}
    by_model = {}
    for vn, sp in view_specs().items():
        by_model.setdefault(sp["model"], []).append(vn)
    EMB = {}
    for mname, vnames in by_model.items():
        try:
            tok = AutoTokenizer.from_pretrained(mname)
            model = AutoModel.from_pretrained(mname).to(DEVICE).eval()
        except Exception as e:
            print(f"[views] could not load '{mname}' ({type(e).__name__}: {str(e)[:120]}) -> skipping {vnames}")
            continue
        for vn in vnames:
            sp = view_specs()[vn]
            t0 = time.time()
            Ex = encode_texts(model, tok, texts[sp["kind"]], sp["max_len"], CFG["enc_bs"])
            at, ay = anchors_for(sp["kind"])
            Ea = encode_texts(model, tok, at, sp["max_len"], CFG["enc_bs"])
            EMB[vn] = dict(all=Ex, anc=Ea, anc_y=ay)
            print(f"[views] {vn:<11} <- {mname}  dim={Ex.shape[1]}  ({time.time() - t0:.0f}s)")
        del model, tok
        free()
    if not EMB:
        raise RuntimeError("No encoder could be loaded. Check internet access / model names in CFG.")
    return EMB


# =====================================================================
# 4. FULL SUPERVISION
# =====================================================================
class ClsNet(nn.Module):
    def __init__(self, name, n_cls=NC, drop=0.1):
        super().__init__()
        self.enc = AutoModel.from_pretrained(name)
        self.drop = nn.Dropout(drop)
        self.head = nn.Linear(self.enc.config.hidden_size, n_cls)

    def forward(self, input_ids, attention_mask):
        h = self.enc(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        return self.head(self.drop(mean_pool(h.float(), attention_mask)))


def tok_pair(tok, A, B, max_len):
    try:
        return tok(A, B, truncation="only_second", max_length=max_len, padding=True, return_tensors="pt")
    except Exception:
        return tok(A, B, truncation=True, max_length=max_len, padding=True, return_tensors="pt")


@torch.inference_mode()
def predict_cls(net, tok, A, B, max_len, bs=64):
    net.eval()
    out = []
    for i in range(0, len(A), bs):
        enc = tok_pair(tok, A[i:i + bs], B[i:i + bs], max_len).to(DEVICE)
        with amp():
            lg = net(enc["input_ids"], enc["attention_mask"])
        out.append(F.softmax(lg.float(), dim=-1).cpu().numpy())
    return np.concatenate(out).astype(np.float64)


def finetune_classifier(name, D, seed):
    """Cross-entropy fine-tuning on (intent+message, diff) pairs; best epoch chosen on VAL."""
    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(name)
    net = ClsNet(name).to(DEVICE)
    A = [f"{i} {m}" for i, m in zip(D["intent"], D["m"])]
    B = [t if t.strip() else "[NO_DIFF]" for t in D["dtxt"]]
    y = D["y"].values
    tr, va, te = [np.where(D["split"].values == s)[0] for s in ("train", "val", "test")]
    ytr, yva = y[tr], y[va]
    cnt = np.bincount(ytr, minlength=NC).astype(float)
    cw = torch.tensor(len(ytr) / (NC * np.maximum(cnt, 1)), dtype=torch.float32, device=DEVICE)

    bs, epochs = CFG["ft_bs"], CFG["ft_epochs"]
    opt = torch.optim.AdamW([{"params": net.enc.parameters(), "lr": CFG["ft_lr"]},
                             {"params": net.head.parameters(), "lr": CFG["ft_lr"] * 20}], weight_decay=0.01)
    total = epochs * math.ceil(len(tr) / bs)
    sch = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
    scaler = make_scaler()
    params = list(net.parameters())
    best = None
    for ep in range(epochs):
        net.train()
        perm = np.random.RandomState(seed * 1000 + ep).permutation(len(tr))
        for i in range(0, len(tr), bs):
            idx = tr[perm[i:i + bs]]
            enc = tok_pair(tok, [A[j] for j in idx], [B[j] for j in idx], CFG["ft_max_len"]).to(DEVICE)
            yt = torch.tensor(y[idx], device=DEVICE)
            with amp():
                logits = net(enc["input_ids"], enc["attention_mask"])
            loss = F.cross_entropy(logits.float(), yt, weight=cw, label_smoothing=0.05)
            backward_step(loss, opt, sch, scaler, params)
        pv = predict_cls(net, tok, [A[j] for j in va], [B[j] for j in va], CFG["ft_max_len"])
        pt = predict_cls(net, tok, [A[j] for j in te], [B[j] for j in te], CFG["ft_max_len"])
        f1 = f1_score(yva, pv.argmax(1), average="macro", zero_division=0)
        nll = log_loss(yva, np.clip(pv, 1e-9, 1), labels=list(range(NC)))
        if ep >= 1 and (best is None or (f1, -nll) > (best["f1"], -best["nll"])):
            best = dict(f1=f1, nll=nll, val=pv, test=pt, ep=ep + 1)
    del net, opt, tok
    free()
    if best is None:   # epochs == 1
        best = dict(f1=f1, nll=nll, val=pv, test=pt, ep=epochs)
    return best


def fit_lr_grid(Xtr, ytr, Xva, yva, Xte, Cs):
    best = None
    for C in Cs:
        clf = LogisticRegression(C=C, max_iter=3000, class_weight="balanced")
        clf.fit(Xtr, ytr)
        pv = _full_proba(clf, Xva)
        nll = log_loss(yva, np.clip(pv, 1e-9, 1), labels=list(range(NC)))
        if best is None or nll < best[0]:
            best = (nll, C, pv, _full_proba(clf, Xte))
    return best[2], best[3], best[1]


def _full_proba(clf, X):
    p = np.zeros((X.shape[0], NC))
    p[:, clf.classes_] = clf.predict_proba(X)
    return p


def fit_weights(pv, yv):
    names = list(pv)
    V = len(names)
    if V == 1:
        return {names[0]: 1.0}
    P = np.stack([pv[n] for n in names])
    ar = np.arange(len(yv))

    def nll(t):
        w = np.exp(t - t.max()); w /= w.sum()
        p = np.tensordot(w, P, axes=1)
        return -np.mean(np.log(p[ar, yv] + 1e-9))

    try:
        w = minimize(nll, np.zeros(V), method="L-BFGS-B").x
        w = np.exp(w - w.max()); w /= w.sum()
    except Exception:
        w = np.ones(V) / V
    w = 0.5 * w + 0.5 / V       # shrink toward uniform: val set is small
    return dict(zip(names, w))


def run_full(D, EMB):
    print("\n" + "=" * 64 + f"\n  FULL SUPERVISION ({CFG['dataset'].upper()})\n" + "=" * 64)
    sp = D["split"].values
    tr, va, te = [np.where(sp == s)[0] for s in ("train", "val", "test")]
    y = D["y"].values
    ytr, yva, yte = y[tr], y[va], y[te]
    PV, PT = {}, {}

    # --- sparse lexical views -------------------------------------------------
    tf = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, token_pattern=r"(?u)\b\w+\b", min_df=1)
    Xm = tf.fit_transform(D["m"].iloc[tr]); 
    PV["tfidf_msg"], PT["tfidf_msg"], _ = fit_lr_grid(Xm, ytr, tf.transform(D["m"].iloc[va]), yva,
                                                      tf.transform(D["m"].iloc[te]), [4, 16, 64])
    docs = D["dtxt"].map(diff_token_doc)
    tf2 = TfidfVectorizer(sublinear_tf=True, token_pattern=r"\S+", min_df=2, max_features=20000)
    try:
        Xd = tf2.fit_transform(docs.iloc[tr])
        PV["tfidf_diff"], PT["tfidf_diff"], _ = fit_lr_grid(Xd, ytr, tf2.transform(docs.iloc[va]), yva,
                                                            tf2.transform(docs.iloc[te]), [1, 4, 16])
    except ValueError:
        print("[full] tfidf_diff skipped (empty vocabulary)")

    # --- structural features --------------------------------------------------
    S = np.array([struct_features(m, d) for m, d in zip(D["message"], D["diff"])], dtype=float)
    sc = StandardScaler().fit(S[tr])
    Ss = np.clip(sc.transform(S), -5, 5)
    rf = RandomForestClassifier(n_estimators=500, min_samples_leaf=2, class_weight="balanced_subsample",
                                random_state=42, n_jobs=-1).fit(S[tr], ytr)
    PV["struct_rf"], PT["struct_rf"] = _full_proba(rf, S[va]), _full_proba(rf, S[te])

    # --- frozen embedding views (+ concatenation with structure) ---------------
    for vn, d in EMB.items():
        X = l2n(d["all"])
        PV[f"emb_{vn}"], PT[f"emb_{vn}"], C = fit_lr_grid(X[tr], ytr, X[va], yva, X[te], CFG["lr_grid"])
    Xc = np.hstack([l2n(d["all"]) for d in EMB.values()] + [Ss * 0.15])
    PV["emb_concat"], PT["emb_concat"], _ = fit_lr_grid(Xc[tr], ytr, Xc[va], yva, Xc[te], CFG["lr_grid"])

    # --- fine-tuned cross-entropy classifiers ---------------------------------
    if CFG["full_ft"]:
        for bb in CFG["ft_backbones"]:
            vs, ts = [], []
            for sd in CFG["ft_seeds"]:
                try:
                    t0 = time.time()
                    r = finetune_classifier(bb, D, sd)
                    vs.append(r["val"]); ts.append(r["test"])
                    print(f"[full] FT {bb.split('/')[-1]} seed={sd}: best epoch={r['ep']} val-F1={r['f1'] * 100:.2f} "
                          f"test-F1={f1_score(yte, r['test'].argmax(1), average='macro') * 100:.2f} ({time.time() - t0:.0f}s)")
                except Exception as e:
                    print(f"[full] fine-tuning {bb} failed: {type(e).__name__}: {str(e)[:150]}")
                    free(); break
            if vs:
                PV[f"ft_{bb.split('/')[-1]}"] = np.mean(vs, axis=0)
                PT[f"ft_{bb.split('/')[-1]}"] = np.mean(ts, axis=0)

    # --- ensemble ---------------------------------------------------------------
    w = fit_weights(PV, yva)
    pv_ens = sum(w[n] * PV[n] for n in PV)
    pt_ens = sum(w[n] * PT[n] for n in PT)
    print(f"\n{'view':<24}{'weight':>8}{'val-F1':>9}{'test-F1':>9}   (test column is diagnostic only)")
    for n in PV:
        print(f"{n:<24}{w[n]:>8.3f}{f1_score(yva, PV[n].argmax(1), average='macro') * 100:>9.2f}"
              f"{f1_score(yte, PT[n].argmax(1), average='macro') * 100:>9.2f}")
    uni = np.mean([PT[n] for n in PT], axis=0)
    print(f"{'ENSEMBLE (fitted)':<24}{'':>8}{f1_score(yva, pv_ens.argmax(1), average='macro') * 100:>9.2f}"
          f"{f1_score(yte, pt_ens.argmax(1), average='macro') * 100:>9.2f}")
    print(f"{'ensemble (uniform)':<24}{'':>8}{'':>9}{f1_score(yte, uni.argmax(1), average='macro') * 100:>9.2f}")
    return metrics(yte, pt_ens.argmax(1))


# =====================================================================
# 5. FEW-SHOT
# =====================================================================
def sample_pairs(items, n_pairs, rng):
    by_label = {c: [i for i, it in enumerate(items) if it["label"] == c] for c in range(NC)}
    real = [i for i, it in enumerate(items) if it["real"]]
    anc = [i for i, it in enumerate(items) if not it["real"]]
    pairs, tries = [], 0
    while len(pairs) < n_pairs and tries < n_pairs * 30:
        tries += 1
        pool = real if (real and (not anc or rng.random() < 0.75)) else anc
        i = rng.choice(pool)
        li = items[i]["label"]
        if rng.random() < 0.5:                       # positive pair
            if not by_label[li]:
                continue
            j = rng.choice(by_label[li])
            ti, tj = rng.choice(items[i]["texts"]), rng.choice(items[j]["texts"])
            if i == j and ti == tj:
                continue
            pairs.append((ti, tj, 1.0))
        else:                                        # negative pair
            cands = [j for c in range(NC) if c != li for j in by_label[c]]
            if not cands:
                continue
            j = rng.choice(cands)
            pairs.append((rng.choice(items[i]["texts"]), rng.choice(items[j]["texts"]), 0.0))
    return pairs


def finetune_bi_encoder(name, items, steps, seed):
    """SetFit-style cosine-similarity fine-tuning with a guaranteed minimum number of steps."""
    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModel.from_pretrained(name).to(DEVICE)
    bs = CFG["fs_con_bs"]
    pairs = sample_pairs(items, steps * bs, random.Random(seed))
    if not pairs:
        return model.eval(), tok
    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["fs_con_lr"], weight_decay=0.01)
    n_steps = math.ceil(len(pairs) / bs)
    sch = get_linear_schedule_with_warmup(opt, int(0.1 * n_steps), n_steps)
    scaler = make_scaler()
    params = list(model.parameters())
    for s in range(n_steps):
        batch = pairs[s * bs:(s + 1) * bs]
        texts = [p[0] for p in batch] + [p[1] for p in batch]
        e = embed_batch(model, tok, texts, CFG["fs_con_max_len"])
        n = len(batch)
        cos = F.cosine_similarity(e[:n], e[n:])
        loss = F.mse_loss(cos, torch.tensor([p[2] for p in batch], device=DEVICE, dtype=cos.dtype))
        backward_step(loss, opt, sch, scaler, params)
    del opt
    return model.eval(), tok


def head_probs(Xs, ys, ws, Xq, k, mu):
    """Centred + L2 features -> weighted prototypes blended with a small LR head."""
    Xs, Xq = l2n(Xs - mu), l2n(Xq - mu)
    P = np.stack([(ws[ys == c, None] * Xs[ys == c]).sum(0) for c in range(NC)])
    p_proto = softmax_np(CFG["tau"] * (Xq @ l2n(P).T))
    lam = min(0.8, k / 12.0)
    if lam < 0.05:
        return p_proto
    clf = LogisticRegression(C=CFG["fs_C"], max_iter=2000)
    clf.fit(Xs, ys, sample_weight=ws)
    return (1 - lam) * p_proto + lam * _full_proba(clf, Xq)


def frozen_view_probs(EMB, train_idx, sup_idx, sup_y, q_idx, k, extra=None):
    out = {}
    for vn, d in EMB.items():
        X = d["all"]
        mu = d.setdefault("mu", X[train_idx].mean(0))
        n_anc = max(1, np.bincount(d["anc_y"], minlength=NC).max())
        w_anc = (CFG["anchor_mass"] / math.sqrt(k)) / n_anc
        Xs = [X[sup_idx], d["anc"]]; ys = [sup_y, d["anc_y"]]
        ws = [np.ones(len(sup_idx)), np.full(len(d["anc_y"]), w_anc)]
        if extra is not None:
            Xs.append(X[extra[0]]); ys.append(extra[1]); ws.append(extra[2])
        out[vn] = head_probs(np.vstack(Xs), np.concatenate(ys), np.concatenate(ws), X[q_idx], k, mu)
    return out


def combine(pdict, lexp, k, weights=None):
    names = list(pdict)
    w = np.array([(weights or {}).get(n, 1.0) for n in names], dtype=float)
    w /= w.sum()
    z = sum(wi * np.log(pdict[n] + 1e-6) for wi, n in zip(w, names))
    if lexp is not None:
        z = z + (CFG["lex_weight"] / (1.0 + k / 5.0)) * np.log(lexp + 1e-6)
    return softmax_np(z)


def contrastive_view_probs(D, EMB, sup_idx, sup_y, train_idx, test_idx, k, seed):
    items = [dict(label=int(l), texts=[D["unified"].iat[i], D["unified_nodiff"].iat[i]], real=True)
             for i, l in zip(sup_idx, sup_y)]
    at, ay = anchors_for("unified")
    items += [dict(label=int(l), texts=[t], real=False) for t, l in zip(at, ay)]
    steps = int(np.clip(24 + 8 * k, 32, 150))
    model, tok = finetune_bi_encoder(CFG["con_backbone"], items, steps, seed)
    ml, bs = CFG["fs_con_max_len"], CFG["enc_bs"]
    rs = np.random.RandomState(seed)
    pool = rs.choice(train_idx, size=min(256, len(train_idx)), replace=False)
    Es = encode_texts(model, tok, D["unified"].iloc[sup_idx].tolist(), ml, bs)
    Ea = encode_texts(model, tok, at, ml, bs)
    Ep = encode_texts(model, tok, D["unified"].iloc[pool].tolist(), ml, bs)
    Eq = encode_texts(model, tok, D["unified"].iloc[test_idx].tolist(), ml, bs)
    del model, tok
    free()
    w_anc = CFG["anchor_mass"] / math.sqrt(k)
    return head_probs(np.vstack([Es, Ea]), np.concatenate([sup_y, ay]),
                      np.concatenate([np.ones(len(sup_idx)), np.full(len(ay), w_anc)]),
                      Eq, k, Ep.mean(0))


def run_fewshot(D, EMB, LEXP):
    sp = D["split"].values
    train_idx, test_idx = np.where(sp == "train")[0], np.where(sp == "test")[0]
    yte = D["y"].values[test_idx]
    tr_df = D.iloc[train_idx]
    train_unique = tr_df.drop_duplicates(subset=["message"])
    res, res_std, diag = {}, {}, {}

    for k in CFG["shots"]:
        print("\n" + "=" * 64 + f"\n  {k}-SHOT ({CFG['dataset'].upper()})\n" + "=" * 64)
        runs, dg = [], {}
        for seed in CFG["fs_seeds"]:
            chunks = [g.sample(n=min(k, len(g)), random_state=seed) for _, g in train_unique.groupby("label")]
            sup = pd.concat(chunks)
            sup_idx, sup_y = sup.index.values, sup["y"].values.astype(int)

            pv = frozen_view_probs(EMB, train_idx, sup_idx, sup_y, test_idx, k)
            weights = {}
            if CFG["self_train"]:
                pool_idx = np.setdiff1d(train_idx, sup_idx)
                pp = combine(frozen_view_probs(EMB, train_idx, sup_idx, sup_y, pool_idx, k), LEXP[pool_idx], k)
                conf, pred = pp.max(1), pp.argmax(1)
                pick = []
                for c in range(NC):
                    ids = np.where(pred == c)[0]
                    pick += list(ids[np.argsort(-conf[ids])][:CFG["self_train_per_class"]])
                pick = np.array(pick, dtype=int)
                if len(pick):
                    extra = (pool_idx[pick], pred[pick], np.full(len(pick), CFG["self_train_w"]))
                    pv = frozen_view_probs(EMB, train_idx, sup_idx, sup_y, test_idx, k, extra=extra)
            if CFG["fs_contrastive"]:
                try:
                    pv["contrastive"] = contrastive_view_probs(D, EMB, sup_idx, sup_y, train_idx, test_idx, k, seed)
                    weights["contrastive"] = CFG["fs_con_weight"]
                except Exception as e:
                    print(f"  [warn] contrastive view failed ({type(e).__name__}: {str(e)[:120]}); frozen views only")
                    free()
            p = combine(pv, LEXP[test_idx], k, weights)
            m = metrics(yte, p.argmax(1))
            runs.append(m)
            for vn, pr in pv.items():
                dg.setdefault(vn, []).append(f1_score(yte, pr.argmax(1), average="macro", zero_division=0))
            dg.setdefault("lexicon-only", []).append(f1_score(yte, LEXP[test_idx].argmax(1), average="macro", zero_division=0))
            print(f"[{k}-Shot seed {seed}] -> Macro F1: {m['F1-score'] * 100:.2f}% | Acc: {m['Accuracy'] * 100:.2f}%")
        res[f"{k}-Shot"] = {key: float(np.mean([r[key] for r in runs])) for key in runs[0]}
        res_std[f"{k}-Shot"] = float(np.std([r["F1-score"] for r in runs]))
        diag[k] = {vn: float(np.mean(v)) for vn, v in dg.items()}
    return res, res_std, diag


# =====================================================================
# 6. ORCHESTRATOR
# =====================================================================
def run_pipeline(dataset_name=None):
    dataset_name = dataset_name or CFG["dataset"]
    CFG["dataset"] = dataset_name
    set_seed(42)
    print(f"Device: {DEVICE}")
    D = add_text_fields(load_dataset_by_name_strict(dataset_name))
    LEXP = lex_probs_from_counts(np.stack([lex_counts(m) for m in D["message"]]), CFG["lex_beta"])
    EMB = compute_frozen_views(D)

    results, stds, diag = {}, {}, {}
    results["Full"] = run_full(D, EMB)
    fs, fs_std, diag = run_fewshot(D, EMB, LEXP)
    results.update(fs); stds.update(fs_std)

    order = ["Full"] + [f"{k}-Shot" for k in CFG["shots"]]
    print("\n\n" + "*" * 86)
    print(f"  DIFFSYNC-ULTRA FINAL RESULTS: {dataset_name.upper()}   (few-shot averaged over {len(CFG['fs_seeds'])} seeds)")
    print("*" * 86)
    print(f"{'Setting':<10} | {'Accuracy':>9} | {'Precision':>9} | {'Recall':>9} | {'F1-score':>9} | {'F1 std':>6} | vs target")
    print("-" * 86)
    rows = []
    for s in order:
        r = results[s]
        tgt = CFG["sota_f1"].get(s)
        flag = "" if tgt is None else (f"{r['F1-score'] * 100 - tgt:+.2f}  {'OK' if r['F1-score'] * 100 > tgt else 'BELOW'}")
        print(f"{s:<10} | {r['Accuracy'] * 100:>8.2f}% | {r['Precision'] * 100:>8.2f}% | {r['Recall'] * 100:>8.2f}% | "
              f"{r['F1-score'] * 100:>8.2f}% | {stds.get(s, 0) * 100:>6.2f} | {flag}")
        rows.append(dict(Setting=s, **{k: v * 100 for k, v in r.items()}, F1_std=stds.get(s, 0) * 100))

    print("\nPer-view macro-F1 (%) in few-shot mode, mean over seeds  [diagnostic; do not tune on it]")
    names = sorted({v for d in diag.values() for v in d})
    print(f"{'view':<14}" + "".join(f"{k}-shot".rjust(9) for k in CFG["shots"]))
    for v in names:
        print(f"{v:<14}" + "".join(f"{diag[k].get(v, float('nan')) * 100:>9.2f}" for k in CFG["shots"]))

    out_dir = CFG["out_dir"] or ("/kaggle/working" if os.path.isdir("/kaggle/working") else ".")
    try:
        path = os.path.join(out_dir, f"diffsync_ultra_{dataset_name}.csv")
        pd.DataFrame(rows).to_csv(path, index=False)
        print(f"\nSaved: {path}")
    except Exception as e:
        print(f"(could not save CSV: {e})")
    return results


if __name__ == "__main__":
    # Change to "multi-lang" to run on the multi-language dataset
    pass
    # run_pipeline(dataset_name="multi-lang")

In [2]:

# ============================================================
# RQ4 — FINAL: MODALITY ROBUSTNESS & NOISE STRESS TESTING
# DiffSync | Ghadhab / Multi-lang
#
# Original DiffSync architecture and hyperparameters unchanged.
# Train on CLEAN data once; evaluate four test conditions.
#
# Conditions:
#   1. Original
#   2. Empty Messages
#   3. Vague Messages
#   4. Deceptive Messages
#
# Outputs:
#   - Robustness metrics
#   - Per-branch robustness
#   - Sample-level predictions
#   - Fixed ensemble weights
#   - Corruption statistics
# ============================================================

import os
import math
import shutil
import tempfile
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    log_loss
)
from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup
)

# ============================================================
# 0. EXPERIMENT CONFIGURATION
# ============================================================

RQ4_DATASET = CFG["dataset"].lower()
RQ4_SEED = 42

RQ4_CONDITIONS = [
    "Original",
    "Empty",
    "Vague",
    "Deceptive"
]

RQ4_OUT = CFG.get("out_dir") or (
    "/kaggle/working"
    if os.path.isdir("/kaggle/working")
    else "."
)

os.makedirs(RQ4_OUT, exist_ok=True)

# Use a unique temporary checkpoint directory.
RQ4_CKPT_DIR = tempfile.mkdtemp(
    prefix=f"diffsync_rq4_{RQ4_DATASET}_"
)

# Keep checkpoints until results are verified.
# Set True only when you intentionally want cleanup.
RQ4_DELETE_CHECKPOINTS = False

# Reference: previously approved Ghadhab Full result.
# This is a diagnostic check, NOT a training target.
RQ4_REFERENCE_F1 = {
    "ghadhab": 86.93
}

RQ4_REFERENCE_TOLERANCE = 0.50

assert RQ4_DATASET in ("ghadhab", "multi-lang")
assert CFG["full_ft"]
assert CFG["ft_epochs"] >= 1
assert len(CFG["ft_seeds"]) > 0

set_seed(RQ4_SEED)

print("=" * 90)
print("RQ4 — MODALITY ROBUSTNESS")
print("Dataset:", RQ4_DATASET.upper())
print("FT Seeds:", CFG["ft_seeds"])
print("Checkpoints:", RQ4_CKPT_DIR)
print("=" * 90)


# ============================================================
# 1. DATASET AND CORRUPTED TEST CONDITIONS
# ============================================================

D = add_text_fields(
    load_dataset_by_name_strict(RQ4_DATASET)
)

sp = D["split"].values

tr, va, te = [
    np.where(sp == s)[0]
    for s in ("train", "val", "test")
]

y = D["y"].values
ytr, yva, yte = y[tr], y[va], y[te]

assert len(tr) > 0
assert len(va) > 0
assert len(te) > 1

assert len(set(tr) & set(va)) == 0
assert len(set(tr) & set(te)) == 0
assert len(set(va) & set(te)) == 0


def rq4_derangement(n, seed):
    """
    Reproducible derangement:
    no sample receives the message from its own index.
    """

    if n < 2:
        raise ValueError("At least two test samples required.")

    rng = np.random.RandomState(seed)
    order = rng.permutation(n)

    mapping = np.empty(n, dtype=int)
    mapping[order] = np.roll(order, 1)

    assert np.all(mapping != np.arange(n))

    return mapping


original_messages = D["message"].iloc[te].tolist()

deceptive_mapping = rq4_derangement(
    len(te), RQ4_SEED
)

RQ4_TESTS = {}

for condition in RQ4_CONDITIONS:

    test_df = D.iloc[te].copy().reset_index(drop=True)

    if condition == "Empty":

        test_df["message"] = "[NO MESSAGE]"

    elif condition == "Vague":

        test_df["message"] = [
            "update" if i % 2 == 0 else "wip"
            for i in range(len(test_df))
        ]

    elif condition == "Deceptive":

        test_df["message"] = [
            original_messages[j]
            for j in deceptive_mapping
        ]

    # Commit diffs and labels must remain unchanged.
    assert np.array_equal(
        test_df["diff"].values,
        D["diff"].iloc[te].values
    )

    assert np.array_equal(
        test_df["y"].values,
        yte
    )

    # Rebuild features after message corruption.
    test_df = add_text_fields(test_df)

    RQ4_TESTS[condition] = test_df


# Verify original condition reproduces clean test rows.
for col in (
    "m", "intent", "dtxt",
    "unified", "unified_nodiff"
):

    assert (
        RQ4_TESTS["Original"][col].tolist()
        == D[col].iloc[te].tolist()
    ), f"Original test preprocessing mismatch: {col}"


n_msg_changed = sum(
    RQ4_TESTS["Deceptive"]["m"].iat[i]
    != RQ4_TESTS["Original"]["m"].iat[i]
    for i in range(len(te))
)

n_cross_class = sum(
    yte[i] != yte[deceptive_mapping[i]]
    for i in range(len(te))
)

RQ4_CORRUPTION_STATS = pd.DataFrame([
    {
        "Dataset": RQ4_DATASET,
        "Test_Samples": len(te),
        "Deceptive_Changed_Messages": n_msg_changed,
        "Deceptive_Changed_Percent":
            100 * n_msg_changed / len(te),
        "Deceptive_Cross_Class_Messages": n_cross_class,
        "Deceptive_Cross_Class_Percent":
            100 * n_cross_class / len(te),
        "Corruption_Seed": RQ4_SEED
    }
])

print("\n[RQ4] Dataset integrity: PASSED")
print("[RQ4] Train:", len(tr))
print("[RQ4] Validation:", len(va))
print("[RQ4] Test:", len(te))
print("[RQ4] Deceptive changed:", n_msg_changed)
print("[RQ4] Cross-class:", n_cross_class)


# ============================================================
# 2. ORIGINAL LOGISTIC REGRESSION GRID
# ============================================================

def rq4_fit_lr(Xtr, Xva, candidates):
    """
    Same Logistic Regression configuration and
    validation-NLL selection used by DiffSync.

    Returns the fitted best estimator.
    """

    best_model = None
    best_nll = float("inf")
    best_C = None

    for C in candidates:

        model = LogisticRegression(
            C=C,
            max_iter=3000,
            class_weight="balanced"
        )

        model.fit(Xtr, ytr)

        pv = _full_proba(model, Xva)

        nll = log_loss(
            yva,
            np.clip(pv, 1e-9, 1),
            labels=list(range(NC))
        )

        if nll < best_nll:
            best_nll = nll
            best_model = model
            best_C = C

    if best_model is None:
        raise RuntimeError("LR model selection failed.")

    return best_model, best_C


# ============================================================
# 3. TRAIN FULL CLASSICAL BRANCHES
# ============================================================

print("\n[RQ4] Computing frozen embeddings...")

EMB = compute_frozen_views(D)

expected_views = set(view_specs())

if set(EMB) != expected_views:
    raise RuntimeError(
        f"Missing frozen views: {expected_views-set(EMB)}"
    )

MODELS = {}
PV = {}
LR_SETTINGS = {}


# ----- Message TF-IDF -----

tf_msg = TfidfVectorizer(
    ngram_range=(1, 2),
    sublinear_tf=True,
    token_pattern=r"(?u)\b\w+\b",
    min_df=1
)

Xm_tr = tf_msg.fit_transform(
    D["m"].iloc[tr]
)

Xm_va = tf_msg.transform(
    D["m"].iloc[va]
)

msg_clf, msg_C = rq4_fit_lr(
    Xm_tr, Xm_va, [4, 16, 64]
)

MODELS["tfidf_msg"] = (tf_msg, msg_clf)
LR_SETTINGS["tfidf_msg"] = msg_C

PV["tfidf_msg"] = _full_proba(
    msg_clf, Xm_va
)


# ----- Diff TF-IDF -----

docs = D["dtxt"].map(diff_token_doc)

tf_diff = TfidfVectorizer(
    sublinear_tf=True,
    token_pattern=r"\S+",
    min_df=2,
    max_features=20000
)

try:

    Xd_tr = tf_diff.fit_transform(
        docs.iloc[tr]
    )

    Xd_va = tf_diff.transform(
        docs.iloc[va]
    )

    diff_clf, diff_C = rq4_fit_lr(
        Xd_tr, Xd_va, [1, 4, 16]
    )

    MODELS["tfidf_diff"] = (
        tf_diff, diff_clf
    )

    LR_SETTINGS["tfidf_diff"] = diff_C

    PV["tfidf_diff"] = _full_proba(
        diff_clf, Xd_va
    )

except ValueError as exc:

    if "empty vocabulary" not in str(exc).lower():
        raise

    print("[RQ4] tfidf_diff skipped: empty vocabulary")


# ----- Structural Features -----

S = np.array(
    [
        struct_features(m, d)
        for m, d in zip(
            D["message"],
            D["diff"]
        )
    ],
    dtype=float
)

struct_scaler = StandardScaler().fit(
    S[tr]
)

Ss = np.clip(
    struct_scaler.transform(S),
    -5, 5
)

struct_rf = RandomForestClassifier(
    n_estimators=500,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

struct_rf.fit(
    S[tr], ytr
)

MODELS["struct_rf"] = struct_rf
MODELS["struct_scaler"] = struct_scaler

PV["struct_rf"] = _full_proba(
    struct_rf, S[va]
)


# ----- Frozen Embedding LR Heads -----

for view_name, view_data in EMB.items():

    X = l2n(view_data["all"])

    clf, chosen_C = rq4_fit_lr(
        X[tr],
        X[va],
        CFG["lr_grid"]
    )

    MODELS[f"emb_{view_name}"] = clf
    LR_SETTINGS[f"emb_{view_name}"] = chosen_C

    PV[f"emb_{view_name}"] = _full_proba(
        clf, X[va]
    )


# ----- Concatenated Embeddings + Structure -----

X_concat = np.hstack(
    [
        l2n(view_data["all"])
        for view_data in EMB.values()
    ]
    + [Ss * 0.15]
)

concat_clf, concat_C = rq4_fit_lr(
    X_concat[tr],
    X_concat[va],
    CFG["lr_grid"]
)

MODELS["emb_concat"] = concat_clf
LR_SETTINGS["emb_concat"] = concat_C

PV["emb_concat"] = _full_proba(
    concat_clf,
    X_concat[va]
)

print("[RQ4] Classical branches: READY")


# ============================================================
# 4. ORIGINAL FINE-TUNING WITH CHECKPOINT RETENTION
# ============================================================

def rq4_train_ft(backbone, seed):
    """
    Original DiffSync FT:
      same model
      same optimizer
      same loss
      same scheduler
      same seed
      same epoch selection

    Only additional operation:
      save the selected checkpoint.

    Model training inputs are CLEAN.
    """

    set_seed(seed)

    tok = AutoTokenizer.from_pretrained(
        backbone
    )

    net = ClsNet(backbone).to(DEVICE)

    A = [
        f"{intent} {msg}"
        for intent, msg in zip(
            D["intent"],
            D["m"]
        )
    ]

    B = [
        text if text.strip() else "[NO_DIFF]"
        for text in D["dtxt"]
    ]

    cnt = np.bincount(
        ytr,
        minlength=NC
    ).astype(float)

    cw = torch.tensor(
        len(ytr) / (
            NC * np.maximum(cnt, 1)
        ),
        dtype=torch.float32,
        device=DEVICE
    )

    bs = CFG["ft_bs"]
    epochs = CFG["ft_epochs"]

    opt = torch.optim.AdamW(
        [
            {
                "params": net.enc.parameters(),
                "lr": CFG["ft_lr"]
            },
            {
                "params": net.head.parameters(),
                "lr": CFG["ft_lr"] * 20
            }
        ],
        weight_decay=0.01
    )

    total_steps = epochs * math.ceil(
        len(tr) / bs
    )

    sch = get_linear_schedule_with_warmup(
        opt,
        int(0.1 * total_steps),
        total_steps
    )

    grad_scaler = make_scaler()
    params = list(net.parameters())

    best = None

    safe_name = backbone.split("/")[-1]

    checkpoint_path = os.path.join(
        RQ4_CKPT_DIR,
        f"{safe_name}_seed{seed}.pt"
    )

    for epoch in range(epochs):

        net.train()

        permutation = np.random.RandomState(
            seed * 1000 + epoch
        ).permutation(len(tr))

        for start in range(0, len(tr), bs):

            idx = tr[
                permutation[start:start + bs]
            ]

            enc = tok_pair(
                tok,
                [A[j] for j in idx],
                [B[j] for j in idx],
                CFG["ft_max_len"]
            ).to(DEVICE)

            yt = torch.tensor(
                y[idx],
                device=DEVICE
            )

            with amp():

                logits = net(
                    enc["input_ids"],
                    enc["attention_mask"]
                )

            loss = F.cross_entropy(
                logits.float(),
                yt,
                weight=cw,
                label_smoothing=0.05
            )

            backward_step(
                loss,
                opt,
                sch,
                grad_scaler,
                params
            )

        # Validation-based epoch selection.
        pv = predict_cls(
            net,
            tok,
            [A[j] for j in va],
            [B[j] for j in va],
            CFG["ft_max_len"]
        )

        # Preserve original test inference during
        # epoch evaluation (not used for selection).
        # This mirrors the original FT workflow.
        _ = predict_cls(
            net,
            tok,
            [A[j] for j in te],
            [B[j] for j in te],
            CFG["ft_max_len"]
        )

        val_f1 = f1_score(
            yva,
            pv.argmax(axis=1),
            average="macro",
            zero_division=0
        )

        val_nll = log_loss(
            yva,
            np.clip(pv, 1e-9, 1),
            labels=list(range(NC))
        )

        if epoch >= 1 and (
            best is None
            or (val_f1, -val_nll)
               > (best["f1"], -best["nll"])
        ):

            best = dict(
                f1=val_f1,
                nll=val_nll,
                val=pv.copy(),
                epoch=epoch + 1
            )

            torch.save(
                net.state_dict(),
                checkpoint_path
            )

        # Original epochs==1 fallback:
        if epochs == 1:

            best = dict(
                f1=val_f1,
                nll=val_nll,
                val=pv.copy(),
                epoch=1
            )

            torch.save(
                net.state_dict(),
                checkpoint_path
            )

    if best is None:
        raise RuntimeError(
            f"No selected checkpoint: {backbone}, {seed}"
        )

    if not os.path.isfile(checkpoint_path):
        raise RuntimeError(
            f"Checkpoint missing: {checkpoint_path}"
        )

    del net, tok, opt, sch, grad_scaler
    free()

    return {
        "val": best["val"],
        "checkpoint": checkpoint_path,
        "epoch": best["epoch"],
        "val_f1": best["f1"]
    }


# ============================================================
# 5. TRAIN FT MODELS ONCE
# ============================================================

FT_CHECKPOINTS = {}

for backbone in CFG["ft_backbones"]:

    key = "ft_" + backbone.split("/")[-1]

    val_predictions = []
    checkpoint_paths = []
    best_epochs = []

    for seed in CFG["ft_seeds"]:

        print(
            f"\n[RQ4 FT] {key} | seed={seed}"
        )

        result = rq4_train_ft(
            backbone,
            seed
        )

        val_predictions.append(
            result["val"]
        )

        checkpoint_paths.append(
            result["checkpoint"]
        )

        best_epochs.append(
            result["epoch"]
        )

        print(
            f"Best validation epoch: "
            f"{result['epoch']}"
        )

    assert len(checkpoint_paths) == len(
        CFG["ft_seeds"]
    )

    PV[key] = np.mean(
        val_predictions,
        axis=0
    )

    FT_CHECKPOINTS[key] = dict(
        backbone=backbone,
        checkpoints=checkpoint_paths,
        selected_epochs=best_epochs
    )


# ============================================================
# 6. FIX ENSEMBLE WEIGHTS ON CLEAN VALIDATION
# ============================================================

RQ4_WEIGHTS = fit_weights(
    PV,
    yva
)

assert set(RQ4_WEIGHTS) == set(PV)

print("\n" + "=" * 75)
print("RQ4 — FIXED CLEAN VALIDATION WEIGHTS")
print("=" * 75)

for name, weight in RQ4_WEIGHTS.items():
    print(f"{name:25s} {weight:.6f}")

print("Total weight:", sum(RQ4_WEIGHTS.values()))


# ============================================================
# 7. FROZEN EMBEDDING INFERENCE
# ============================================================

def rq4_encode_frozen(test_df):

    specs = view_specs()
    result = {}

    model_names = list(dict.fromkeys(
        spec["model"]
        for spec in specs.values()
    ))

    for model_name in model_names:

        tok = AutoTokenizer.from_pretrained(
            model_name
        )

        net = AutoModel.from_pretrained(
            model_name
        ).to(DEVICE).eval()

        for view_name, spec in specs.items():

            if spec["model"] != model_name:
                continue

            kind = spec["kind"]

            if kind == "msg":

                texts = test_df["m"].tolist()

            elif kind == "diff":

                texts = [
                    t if t.strip() else "[NO_DIFF]"
                    for t in test_df["dtxt"]
                ]

            elif kind == "unified":

                texts = test_df["unified"].tolist()

            else:

                raise ValueError(
                    f"Unknown view kind: {kind}"
                )

            result[view_name] = encode_texts(
                net,
                tok,
                texts,
                spec["max_len"],
                CFG["enc_bs"]
            )

        del net, tok
        free()

    assert set(result) == set(EMB)

    return result


# ============================================================
# 8. FT INFERENCE USING SAVED CHECKPOINTS
# ============================================================

def rq4_predict_ft(test_df):

    outputs = {}

    A = [
        f"{intent} {msg}"
        for intent, msg in zip(
            test_df["intent"],
            test_df["m"]
        )
    ]

    B = [
        text if text.strip() else "[NO_DIFF]"
        for text in test_df["dtxt"]
    ]

    for key, info in FT_CHECKPOINTS.items():

        predictions = []

        for checkpoint in info["checkpoints"]:

            if not os.path.isfile(checkpoint):
                raise FileNotFoundError(checkpoint)

            tok = AutoTokenizer.from_pretrained(
                info["backbone"]
            )

            net = ClsNet(
                info["backbone"]
            ).to(DEVICE)

            state = torch.load(
                checkpoint,
                map_location="cpu",
                weights_only=True
            )

            net.load_state_dict(state)

            probs = predict_cls(
                net,
                tok,
                A,
                B,
                CFG["ft_max_len"]
            )

            predictions.append(probs)

            del net, tok, state
            free()

        outputs[key] = np.mean(
            predictions,
            axis=0
        )

    return outputs


# ============================================================
# 9. COMPLETE FIXED-MODEL INFERENCE
# ============================================================

def rq4_predict_condition(condition):

    test_df = RQ4_TESTS[condition]

    PT = {}

    # Message TF-IDF
    msg_vec, msg_clf = MODELS["tfidf_msg"]

    PT["tfidf_msg"] = _full_proba(
        msg_clf,
        msg_vec.transform(test_df["m"])
    )

    # Diff TF-IDF
    if "tfidf_diff" in MODELS:

        diff_vec, diff_clf = MODELS["tfidf_diff"]

        PT["tfidf_diff"] = _full_proba(
            diff_clf,
            diff_vec.transform(
                test_df["dtxt"].map(diff_token_doc)
            )
        )

    # Structural features
    St = np.array(
        [
            struct_features(m, d)
            for m, d in zip(
                test_df["message"],
                test_df["diff"]
            )
        ],
        dtype=float
    )

    Sst = np.clip(
        struct_scaler.transform(St),
        -5, 5
    )

    PT["struct_rf"] = _full_proba(
        struct_rf, St
    )

    # Frozen transformer embeddings
    E = rq4_encode_frozen(test_df)

    for view_name, embedding in E.items():

        clf = MODELS[f"emb_{view_name}"]

        PT[f"emb_{view_name}"] = _full_proba(
            clf,
            l2n(embedding)
        )

    # Concatenated representation
    test_concat = np.hstack(
        [
            l2n(E[name])
            for name in EMB.keys()
        ]
        + [Sst * 0.15]
    )

    PT["emb_concat"] = _full_proba(
        concat_clf,
        test_concat
    )

    # Fine-tuned transformers (fixed weights)
    PT.update(
        rq4_predict_ft(test_df)
    )

    if set(PT) != set(RQ4_WEIGHTS):

        raise RuntimeError(
            "Prediction branch mismatch!\n"
            f"Missing: {set(RQ4_WEIGHTS)-set(PT)}\n"
            f"Extra: {set(PT)-set(RQ4_WEIGHTS)}"
        )

    probs = sum(
        RQ4_WEIGHTS[name] * PT[name]
        for name in RQ4_WEIGHTS
    )

    assert probs.shape == (len(te), NC)
    assert np.all(np.isfinite(probs))

    return probs, PT


# ============================================================
# 10. EVALUATE ALL FOUR TEST CONDITIONS
# ============================================================

RQ4_RESULTS = {}
RQ4_BRANCH_RESULTS = {}
RQ4_PREDICTIONS = []

for condition in RQ4_CONDITIONS:

    print("\n" + "-" * 75)
    print("RQ4 Condition:", condition)
    print("-" * 75)

    probs, branch_probs = rq4_predict_condition(
        condition
    )

    predictions = probs.argmax(axis=1)

    acc = 100 * accuracy_score(
        yte, predictions
    )

    f1 = 100 * f1_score(
        yte,
        predictions,
        average="macro",
        zero_division=0
    )

    prec = 100 * precision_score(
        yte,
        predictions,
        average="macro",
        zero_division=0
    )

    rec = 100 * recall_score(
        yte,
        predictions,
        average="macro",
        zero_division=0
    )

    RQ4_RESULTS[condition] = dict(
        Accuracy=acc,
        Macro_F1=f1,
        Macro_Precision=prec,
        Macro_Recall=rec
    )

    RQ4_BRANCH_RESULTS[condition] = branch_probs

    for j, original_idx in enumerate(te):

        row = dict(
            Dataset=RQ4_DATASET,
            Condition=condition,
            Test_Index=int(original_idx),
            True_Label=int(yte[j]),
            Pred_Label=int(predictions[j])
        )

        for class_idx in range(NC):
            row[f"P_{CLASSES[class_idx]}"] = float(
                probs[j, class_idx]
            )

        RQ4_PREDICTIONS.append(row)

    print(f"Accuracy : {acc:.3f}%")
    print(f"Macro F1 : {f1:.3f}%")


# ============================================================
# 11. BASELINE VERIFICATION
# ============================================================

original_f1 = RQ4_RESULTS["Original"]["Macro_F1"]
original_acc = RQ4_RESULTS["Original"]["Accuracy"]

reference_f1 = RQ4_REFERENCE_F1.get(
    RQ4_DATASET
)

baseline_difference = np.nan
baseline_status = "NO_REFERENCE"

if reference_f1 is not None:

    baseline_difference = (
        original_f1 - reference_f1
    )

    baseline_status = (
        "PASS"
        if abs(baseline_difference)
        <= RQ4_REFERENCE_TOLERANCE
        else "WARNING"
    )

    print("\n" + "=" * 75)
    print("RQ4 — BASELINE VERIFICATION")
    print("=" * 75)

    print(f"Reference F1 : {reference_f1:.3f}%")
    print(f"RQ4 F1       : {original_f1:.3f}%")
    print(f"Difference   : {baseline_difference:+.3f} pp")
    print(f"Status       : {baseline_status}")

    if baseline_status == "WARNING":
        print(
            "[WARNING] Original baseline differs from "
            "previously approved Full result. "
            "Results will be saved, but this "
            "difference must be reported."
        )


# ============================================================
# 12. FINAL RQ4 ROBUSTNESS TABLE
# ============================================================

rows = []

for condition, result in RQ4_RESULTS.items():

    acc_drop = (
        original_acc - result["Accuracy"]
    )

    f1_drop = (
        original_f1 - result["Macro_F1"]
    )

    acc_relative_drop = (
        100 * acc_drop / original_acc
        if original_acc > 0 else np.nan
    )

    f1_relative_drop = (
        100 * f1_drop / original_f1
        if original_f1 > 0 else np.nan
    )

    rows.append(dict(
        Dataset=RQ4_DATASET,
        Condition=condition,
        Accuracy=result["Accuracy"],
        Macro_F1=result["Macro_F1"],
        Macro_Precision=result["Macro_Precision"],
        Macro_Recall=result["Macro_Recall"],
        Accuracy_Drop_pp=acc_drop,
        Macro_F1_Drop_pp=f1_drop,
        Accuracy_Relative_Drop_pct=acc_relative_drop,
        Macro_F1_Relative_Drop_pct=f1_relative_drop,
        Accuracy_Relative_Drop_Under_15pct=(
            acc_relative_drop < 15
            if condition != "Original"
            else np.nan
        )
    ))

RQ4_TABLE = pd.DataFrame(rows)

print("\n" + "=" * 105)
print("RQ4 — MODALITY ROBUSTNESS RESULTS")
print("=" * 105)

print(
    RQ4_TABLE.round(3).to_string(index=False)
)


# ============================================================
# 13. PER-BRANCH ROBUSTNESS
# ============================================================

branch_rows = []

for branch_name, weight in RQ4_WEIGHTS.items():

    original_branch_probs = (
        RQ4_BRANCH_RESULTS["Original"][branch_name]
    )

    original_branch_f1 = 100 * f1_score(
        yte,
        original_branch_probs.argmax(axis=1),
        average="macro",
        zero_division=0
    )

    row = dict(
        Dataset=RQ4_DATASET,
        Branch=branch_name,
        Ensemble_Weight=weight
    )

    for condition in RQ4_CONDITIONS:

        branch_probs = (
            RQ4_BRANCH_RESULTS[condition][branch_name]
        )

        branch_pred = branch_probs.argmax(axis=1)

        branch_f1 = 100 * f1_score(
            yte,
            branch_pred,
            average="macro",
            zero_division=0
        )

        row[f"{condition}_F1"] = branch_f1

        row[f"{condition}_Drop_pp"] = (
            original_branch_f1 - branch_f1
        )

    branch_rows.append(row)

RQ4_BRANCH_TABLE = pd.DataFrame(
    branch_rows
)

print("\n" + "=" * 105)
print("RQ4 — PER-BRANCH ROBUSTNESS")
print("=" * 105)

print(
    RQ4_BRANCH_TABLE.round(3).to_string(index=False)
)


# ============================================================
# 14. SAVE ALL RESULTS
# ============================================================

RQ4_PATHS = {
    "robustness": os.path.join(
        RQ4_OUT,
        f"rq4_{RQ4_DATASET}_robustness.csv"
    ),
    "branches": os.path.join(
        RQ4_OUT,
        f"rq4_{RQ4_DATASET}_branch_robustness.csv"
    ),
    "predictions": os.path.join(
        RQ4_OUT,
        f"rq4_{RQ4_DATASET}_predictions.csv"
    ),
    "weights": os.path.join(
        RQ4_OUT,
        f"rq4_{RQ4_DATASET}_fixed_weights.csv"
    ),
    "corruption": os.path.join(
        RQ4_OUT,
        f"rq4_{RQ4_DATASET}_corruption_stats.csv"
    ),
    "baseline": os.path.join(
        RQ4_OUT,
        f"rq4_{RQ4_DATASET}_baseline_check.csv"
    ),
}

RQ4_TABLE.to_csv(
    RQ4_PATHS["robustness"],
    index=False
)

RQ4_BRANCH_TABLE.to_csv(
    RQ4_PATHS["branches"],
    index=False
)

pd.DataFrame(RQ4_PREDICTIONS).to_csv(
    RQ4_PATHS["predictions"],
    index=False
)

pd.DataFrame([
    {
        "Dataset": RQ4_DATASET,
        "Branch": name,
        "Weight": weight
    }
    for name, weight in RQ4_WEIGHTS.items()
]).to_csv(
    RQ4_PATHS["weights"],
    index=False
)

RQ4_CORRUPTION_STATS.to_csv(
    RQ4_PATHS["corruption"],
    index=False
)

pd.DataFrame([{
    "Dataset": RQ4_DATASET,
    "Original_F1": original_f1,
    "Reference_F1": reference_f1,
    "Difference_pp": baseline_difference,
    "Tolerance_pp": RQ4_REFERENCE_TOLERANCE,
    "Status": baseline_status
}]).to_csv(
    RQ4_PATHS["baseline"],
    index=False
)


# ============================================================
# 15. CHECKPOINT MANAGEMENT
# ============================================================

if RQ4_DELETE_CHECKPOINTS:

    shutil.rmtree(
        RQ4_CKPT_DIR,
        ignore_errors=False
    )

    print("\n[RQ4] Temporary checkpoints removed.")

else:

    print("\n[RQ4] Checkpoints retained:")
    print(RQ4_CKPT_DIR)

    print(
        "[RQ4] Note: /tmp storage is temporary "
        "and may be cleared when the session ends."
    )


# ============================================================
# 16. FINAL STATUS
# ============================================================

print("\n" + "=" * 90)
print("RQ4 EXPERIMENT COMPLETED")
print("=" * 90)

print("Dataset:", RQ4_DATASET.upper())
print("Test conditions:", RQ4_CONDITIONS)
print("Baseline status:", baseline_status)

print("\nMethodological checks:")
print("  Clean training only: YES")
print("  Corrupted test only: YES")
print("  Same fitted models: YES")
print("  Fixed ensemble weights: YES")
print("  No corrupted-test retraining: YES")

print("\nSaved outputs:")

for name, path in RQ4_PATHS.items():
    print(f"  {name}: {path}")

print("\nOriginal model source code was not modified.")


RQ4 — MODALITY ROBUSTNESS
Dataset: MULTI-LANG
FT Seeds: [42, 43, 44]
Checkpoints: /tmp/diffsync_rq4_multi-lang_2rqjyjah
[data] file=/kaggle/input/datasets/zohrehsadatsajjadi/multi-lang-annotated/multi-lang-annotated.csv
[data] columns -> message='commit_message', diff='commit_diff', label='maintenance_type'

--- MULTI-LANG loaded: 1581 commits | {'Perfective': 989, 'Corrective': 353, 'Adaptive': 239} ---
[data] split sizes: train=1106 val=237 test=238

[RQ4] Dataset integrity: PASSED
[RQ4] Train: 1106
[RQ4] Validation: 237
[RQ4] Test: 238
[RQ4] Deceptive changed: 238
[RQ4] Cross-class: 123

[RQ4] Computing frozen embeddings...


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

... (72 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Accuracy : 55.462%
Macro F1 : 40.443%

RQ4 — MODALITY ROBUSTNESS RESULTS
   Dataset Condition  Accuracy  Macro_F1  Macro_Precision  Macro_Recall  Accuracy_Drop_pp  Macro_F1_Drop_pp  Accuracy_Relative_Drop_pct  Macro_F1_Relative_Drop_pct Accuracy_Relative_Drop_Under_15pct
multi-lang  Original    87.815    85.999           83.887        88.602             0.000             0.000                       0.000                       0.000                                NaN
multi-lang     Empty    70.588    49.338           71.437        47.853            17.227            36.661                      19.617                      42.629                              False
multi-lang     Vague    66.807    43.493           69.548        46.255            21.008            42.506                      23.923                      49.426                              False
multi-lang Deceptive    55.462    40.443           40.675        40.262            32.353            45.556                      36

In [3]:

# ============================================================
# RQ7 — COMPUTATIONAL PROFILING
# DiffSync | Ghadhab / Multi-lang
#
# Requires fitted RQ4 state in the same Kaggle session.
# No changes to model architecture or weights.
# ============================================================

import os
import gc
import time
import platform
import subprocess
import numpy as np
import pandas as pd
import torch
import psutil

from transformers import AutoModel, AutoTokenizer

# ============================================================
# 0. SETTINGS
# ============================================================

RQ7_DATASET = CFG["dataset"].lower()

# Number of test commits used for profiling.
RQ7_N = 20

# One warmup inference before measuring each neural model.
RQ7_WARMUP = 1

# Set True only if an actual Full training time measurement
# is required. This reruns the ORIGINAL Full training pipeline.
RQ7_RUN_TRAINING = True

RQ7_OUT = CFG.get("out_dir") or (
    "/kaggle/working"
    if os.path.isdir("/kaggle/working") else "."
)
os.makedirs(RQ7_OUT, exist_ok=True)

REQUIRED_RQ4 = [
    "RQ4_TESTS",
    "MODELS",
    "FT_CHECKPOINTS",
    "RQ4_WEIGHTS",
    "EMB",
    "struct_scaler",
    "struct_rf"
]

missing = [x for x in REQUIRED_RQ4 if x not in globals()]
if missing:
    raise RuntimeError(
        "RQ4 fitted state is unavailable. "
        "Run RQ4 in this session first. Missing: "
        + ", ".join(missing)
    )

assert RQ7_DATASET == RQ4_DATASET
assert RQ7_N >= 2
assert RQ7_WARMUP >= 1
assert all(
    os.path.isfile(path)
    for info in FT_CHECKPOINTS.values()
    for path in info["checkpoints"]
), "Some RQ4 checkpoints are missing."

RQ7_TEST = RQ4_TESTS["Original"].reset_index(drop=True)
RQ7_N = min(RQ7_N, len(RQ7_TEST))

# Reproducible representative sample.
rng = np.random.RandomState(42)
sample_ids = rng.choice(
    len(RQ7_TEST), size=RQ7_N, replace=False
)

process = psutil.Process(os.getpid())

def rq7_ram_mb():
    return process.memory_info().rss / (1024 ** 2)

def rq7_sync(device):
    if device == "cuda":
        torch.cuda.synchronize()

def rq7_measure(fn, device):
    rq7_sync(device)
    start = time.perf_counter()
    result = fn()
    rq7_sync(device)
    return (time.perf_counter() - start) * 1000, result

def rq7_tensor_device(device):
    return torch.device(device)

def rq7_free():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Save original global inference settings.
RQ7_ORIGINAL_DEVICE = DEVICE
RQ7_ORIGINAL_AMP = USE_AMP

RQ7_DEVICES = ["cpu"]
if torch.cuda.is_available():
    RQ7_DEVICES.append("cuda")

# ============================================================
# 1. HARDWARE METADATA
# ============================================================

hardware = {
    "Dataset": RQ7_DATASET,
    "OS": platform.platform(),
    "Python": platform.python_version(),
    "PyTorch": torch.__version__,
    "CPU": platform.processor(),
    "Logical_CPU_Count": psutil.cpu_count(),
    "Total_RAM_MB": psutil.virtual_memory().total / (1024**2),
    "GPU_Available": torch.cuda.is_available(),
    "GPU": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available() else "N/A"
    ),
    "CUDA_Version": torch.version.cuda,
    "Test_Samples": len(RQ7_TEST),
    "Profiled_Samples": RQ7_N,
    "Batch_Size": 1,
    "FT_Seeds": str(CFG["ft_seeds"])
}

pd.DataFrame([hardware]).to_csv(
    os.path.join(RQ7_OUT, f"rq7_{RQ7_DATASET}_hardware.csv"),
    index=False
)

print("RQ7 HARDWARE")
for k, v in hardware.items():
    print(f"{k}: {v}")

# ============================================================
# 2. PREPARE TEXT INPUTS
# ============================================================

samples = RQ7_TEST.iloc[sample_ids].reset_index(drop=True)

A = [
    f"{i} {m}"
    for i, m in zip(samples["intent"], samples["m"])
]

B = [
    t if t.strip() else "[NO_DIFF]"
    for t in samples["dtxt"]
]

specs = view_specs()

texts = {
    "msg": samples["m"].tolist(),
    "diff": [
        t if t.strip() else "[NO_DIFF]"
        for t in samples["dtxt"]
    ],
    "unified": samples["unified"].tolist()
}

# Per-device, per-commit measured latency.
RQ7_RECORDS = []
RQ7_MEMORY = []

# ============================================================
# 3. PROFILE COMPLETE SERIAL INFERENCE COMPONENTS
# ============================================================

def rq7_profile_device(device):

    global DEVICE, USE_AMP

    DEVICE = torch.device(device)
    USE_AMP = (device == "cuda" and RQ7_ORIGINAL_AMP)

    # Accumulate per-sample times for ALL branches.
    times = {i: 0.0 for i in range(RQ7_N)}
    components = []

    rss_start = rq7_ram_mb()
    rss_peak_observed = rss_start
    gpu_peak = 0.0
    gpu_reserved_peak = 0.0

    def update_memory():
        nonlocal rss_peak_observed, gpu_peak, gpu_reserved_peak
        rss_peak_observed = max(rss_peak_observed, rq7_ram_mb())

        if device == "cuda":
            gpu_peak = max(
                gpu_peak,
                torch.cuda.max_memory_allocated() / (1024**2)
            )
            gpu_reserved_peak = max(
                gpu_reserved_peak,
                torch.cuda.max_memory_reserved() / (1024**2)
            )

    def record(branch, latencies):
        for i, ms in enumerate(latencies):
            times[i] += ms
        components.append({
            "Device": device.upper(),
            "Component": branch,
            "Mean_ms": float(np.mean(latencies)),
            "Median_ms": float(np.median(latencies)),
            "P95_ms": float(np.percentile(latencies, 95))
        })

    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
    update_memory()

    # --------------------------------------------------------
    # 3.1 Preprocessing: original feature construction
    # --------------------------------------------------------

    preprocess_times = []
    struct_rows = []
    token_docs = []

    for _, row in samples.iterrows():

        def prepare():
            return (
                struct_features(row["message"], row["diff"]),
                diff_token_doc(row["dtxt"])
            )

        ms, (feat, doc) = rq7_measure(prepare, device)
        preprocess_times.append(ms)
        struct_rows.append(feat)
        token_docs.append(doc)

    record("Preprocessing", preprocess_times)
    update_memory()

    # --------------------------------------------------------
    # 3.2 Classical models (batch size 1)
    # --------------------------------------------------------

    classical_times = []

    for i in range(RQ7_N):

        def classical_one():
            m = samples["m"].iat[i]
            d = token_docs[i]
            s = np.asarray([struct_rows[i]], dtype=float)

            msg_vectorizer, msg_clf = MODELS["tfidf_msg"]
            _ = msg_clf.predict_proba(
                msg_vectorizer.transform([m])
            )

            if "tfidf_diff" in MODELS:
                diff_vectorizer, diff_clf = MODELS["tfidf_diff"]
                _ = diff_clf.predict_proba(
                    diff_vectorizer.transform([d])
                )

            _ = struct_rf.predict_proba(s)
            _ = struct_scaler.transform(s)

        ms, _ = rq7_measure(classical_one, device)
        classical_times.append(ms)
        update_memory()

    record("TFIDF_RF_Structural", classical_times)

    # --------------------------------------------------------
    # 3.3 Frozen encoders, measured one model at a time
    # --------------------------------------------------------

    frozen_models = list(dict.fromkeys(
        sp["model"] for sp in specs.values()
    ))

    frozen_embs = {}

    for name in frozen_models:

        tok = AutoTokenizer.from_pretrained(name)
        net = AutoModel.from_pretrained(name).to(DEVICE).eval()

        relevant = [
            (vn, spec)
            for vn, spec in specs.items()
            if spec["model"] == name
        ]

        for vn, spec in relevant:

            inputs = texts[spec["kind"]]

            def encode_one(index):
                return encode_texts(
                    net, tok,
                    [inputs[index]],
                    spec["max_len"],
                    1
                )

            # Warm up on the same input without recording timing.
            for _ in range(RQ7_WARMUP):
                encode_one(0)

            latencies = []
            collected = []

            for i in range(RQ7_N):
                ms, emb = rq7_measure(
                    lambda i=i: encode_one(i),
                    device
                )
                latencies.append(ms)
                collected.append(emb)
                update_memory()

            frozen_embs[vn] = np.vstack(collected)
            record(f"Frozen_{vn}", latencies)

        del net, tok
        rq7_free()
        update_memory()

    # --------------------------------------------------------
    # 3.4 Frozen embedding LR + concatenation + ensemble
    # --------------------------------------------------------

    head_times = []

    for i in range(RQ7_N):

        def heads_one():
            s = np.asarray([struct_rows[i]], dtype=float)
            ss = np.clip(struct_scaler.transform(s), -5, 5)

            for vn in EMB.keys():
                x = l2n(frozen_embs[vn][i:i+1])
                _ = MODELS[f"emb_{vn}"].predict_proba(x)

            xc = np.hstack(
                [l2n(frozen_embs[vn][i:i+1]) for vn in EMB.keys()]
                + [ss * 0.15]
            )
            _ = MODELS["emb_concat"].predict_proba(xc)

            # Weighted ensemble itself
            # (10 probability vectors, 3 classes).
            dummy = np.full((1, NC), 1 / NC, dtype=float)
            _ = sum(w * dummy for w in RQ4_WEIGHTS.values())

        ms, _ = rq7_measure(heads_one, device)
        head_times.append(ms)
        update_memory()

    record("Embedding_Heads_And_Ensemble", head_times)

    # --------------------------------------------------------
    # 3.5 Fine-tuned classifiers — ALL saved seeds
    # --------------------------------------------------------

    for branch, info in FT_CHECKPOINTS.items():

        for seed_idx, ckpt in enumerate(info["checkpoints"]):

            tok = AutoTokenizer.from_pretrained(info["backbone"])
            net = ClsNet(info["backbone"]).to(DEVICE)

            state = torch.load(
                ckpt,
                map_location="cpu",
                weights_only=True
            )
            net.load_state_dict(state)
            net.eval()

            del state
            update_memory()

            def ft_one(index):
                return predict_cls(
                    net, tok,
                    [A[index]], [B[index]],
                    CFG["ft_max_len"],
                    bs=1
                )

            for _ in range(RQ7_WARMUP):
                ft_one(0)

            latencies = []

            for i in range(RQ7_N):
                ms, _ = rq7_measure(
                    lambda i=i: ft_one(i),
                    device
                )
                latencies.append(ms)
                update_memory()

            record(f"{branch}_seed{seed_idx}", latencies)

            del net, tok
            rq7_free()
            update_memory()

    # --------------------------------------------------------
    # 3.6 Save memory and per-commit serial latency
    # --------------------------------------------------------

    # The peak is across sequential stages, not all models
    # loaded concurrently.
    RQ7_MEMORY.append({
        "Device": device.upper(),
        "RAM_RSS_Start_MB": rss_start,
        "RAM_RSS_Peak_Observed_MB": rss_peak_observed,
        "RAM_RSS_Observed_Increase_MB":
            max(0, rss_peak_observed - rss_start),
        "GPU_Peak_Allocated_MB":
            gpu_peak if device == "cuda" else np.nan,
        "GPU_Peak_Reserved_MB":
            gpu_reserved_peak if device == "cuda" else np.nan,
        "Memory_Interpretation": "Sequential stage-wise peak"
    })

    for i, ms in times.items():
        RQ7_RECORDS.append({
            "Dataset": RQ7_DATASET,
            "Device": device.upper(),
            "Sample_ID": int(sample_ids[i]),
            "Batch_Size": 1,
            "Serial_Component_Sum_ms": float(ms)
        })

    return pd.DataFrame(components)


# ============================================================
# 4. RUN CPU AND GPU
# ============================================================

RQ7_COMPONENT_TABLES = []

try:
    for dev in RQ7_DEVICES:
        print(f"\n[RQ7] Profiling {dev.upper()} ...")
        component_df = rq7_profile_device(dev)
        RQ7_COMPONENT_TABLES.append(component_df)
        print(component_df.to_string(index=False))
finally:
    # Restore the original notebook global inference settings.
    DEVICE = RQ7_ORIGINAL_DEVICE
    USE_AMP = RQ7_ORIGINAL_AMP
    rq7_free()

RQ7_LATENCY_RAW = pd.DataFrame(RQ7_RECORDS)
RQ7_MEMORY_TABLE = pd.DataFrame(RQ7_MEMORY)
RQ7_COMPONENT_TABLE = pd.concat(
    RQ7_COMPONENT_TABLES,
    ignore_index=True
)

RQ7_LATENCY = (
    RQ7_LATENCY_RAW
    .groupby("Device", as_index=False)
    .agg(
        Mean_ms=("Serial_Component_Sum_ms", "mean"),
        Median_ms=("Serial_Component_Sum_ms", "median"),
        P95_ms=(
            "Serial_Component_Sum_ms",
            lambda x: np.percentile(x, 95)
        ),
        Min_ms=("Serial_Component_Sum_ms", "min"),
        Max_ms=("Serial_Component_Sum_ms", "max"),
        N_Commits=("Sample_ID", "count")
    )
)

# ============================================================
# 5. TRAINING TIME — EXACT ORIGINAL FULL PIPELINE
# ============================================================

RQ7_TRAINING_ROWS = []

if RQ7_RUN_TRAINING:

    print("\n[RQ7] Measuring original Full training time...")

    # This reruns the original Full training, not a modified model.
    # Includes frozen encoding + fitting all Full branches,
    # including the original FT seeds and validation operations.

    set_seed(42)

    train_D = add_text_fields(
        load_dataset_by_name_strict(RQ7_DATASET)
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    t0 = time.perf_counter()
    ram_before = rq7_ram_mb()

    frozen = compute_frozen_views(train_D)
    _ = run_full(train_D, frozen)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - t0

    RQ7_TRAINING_ROWS.append({
        "Dataset": RQ7_DATASET,
        "Training_Protocol": "Original Full Supervision",
        "Training_Seconds": elapsed,
        "Training_Minutes": elapsed / 60,
        "Includes_Frozen_Encoding": True,
        "Includes_FT_Seeds": str(CFG["ft_seeds"]),
        "Includes_Validation_And_Test_Evaluation": True,
        "RAM_RSS_Before_MB": ram_before,
        "RAM_RSS_After_MB": rq7_ram_mb()
    })

    del frozen, train_D
    rq7_free()

RQ7_TRAINING = pd.DataFrame(RQ7_TRAINING_ROWS)

# ============================================================
# 6. FINAL REPORT
# ============================================================

print("\n" + "=" * 90)
print("RQ7 — INFERENCE LATENCY (SERIAL COMPONENT SUM)")
print("=" * 90)
print(RQ7_LATENCY.round(3).to_string(index=False))

print("\n" + "=" * 90)
print("RQ7 — PEAK MEMORY")
print("=" * 90)
print(RQ7_MEMORY_TABLE.round(2).to_string(index=False))

print("\n" + "=" * 90)
print("RQ7 — FULL TRAINING TIME")
print("=" * 90)
if not RQ7_TRAINING.empty:
    print(RQ7_TRAINING.round(3).to_string(index=False))
else:
    print("Not measured; RQ7_RUN_TRAINING=False")

# ============================================================
# 7. SAVE
# ============================================================

RQ7_OUTPUTS = {
    "latency": RQ7_LATENCY,
    "latency_samples": RQ7_LATENCY_RAW,
    "components": RQ7_COMPONENT_TABLE,
    "memory": RQ7_MEMORY_TABLE,
    "training": RQ7_TRAINING
}

for name, df in RQ7_OUTPUTS.items():
    path = os.path.join(
        RQ7_OUT, f"rq7_{RQ7_DATASET}_{name}.csv"
    )
    df.to_csv(path, index=False)
    print("[Saved]", path)

print("\nRQ7 COMPLETE")
print("Original model configuration restored.")
print("No model architecture changes.")
print("Batch Size = 1.")
print("GPU synchronization enabled.")
print("Latency is serial component-sum, excluding model loading.")
print("Memory is observed sequential peak, not total resident model size.")


RQ7 HARDWARE
Dataset: multi-lang
OS: Linux-6.18.48+-x86_64-with-glibc2.39
Python: 3.13.15
PyTorch: 2.11.0+cu128
CPU: x86_64
Logical_CPU_Count: 4
Total_RAM_MB: 32099.0703125
GPU_Available: True
GPU: Tesla T4
CUDA_Version: 12.8
Test_Samples: 238
Profiled_Samples: 20
Batch_Size: 1
FT_Seeds: [42, 43, 44]

[RQ7] Profiling CPU ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

... (29 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[full] FT all-mpnet-base-v2 seed=43: best epoch=4 val-F1=85.50 test-F1=83.74 (298s)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[full] FT all-mpnet-base-v2 seed=44: best epoch=6 val-F1=84.72 test-F1=87.68 (298s)

view                      weight   val-F1  test-F1   (test column is diagnostic only)
tfidf_msg                  0.050    74.37    76.58
tfidf_diff                 0.050    47.08    50.84
struct_rf                  0.050    77.06    75.73
emb_msg_all                0.050    65.26    65.48
emb_msg_para               0.050    71.82    73.06
emb_diff_code              0.050    50.13    49.45
emb_joint_code             0.050    63.47    60.73
emb_concat                 0.089    77.18    78.06
ft_unixcoder-base          0.204    83.19    84.84
ft_all-mpnet-base-v2       0.357    85.04    86.88
ENSEMBLE (fitted)                   84.98    86.53
ensemble (uniform)                           83.35

RQ7 — INFERENCE LATENCY (SERIAL COMPONENT SUM)
Device  Mean_ms  Median_ms   P95_ms  Min_ms   Max_ms  N_Commits
   CPU 2693.452   3097.390 3181.372 880.282 3231.309         20
  CUDA  253.507    253.027  271.505 229.5